In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 31))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from scripts.weighted_coverage import wknn_one
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import create_point_matrix, compute_weights
import pandas as pd


def compute_weights_pca(m_rfp_pca, m_tp_pca):
    """
    Compute weights using PCA-transformed data
    """
    # Compute Euclidean distances in the PCA space
    D = cdist(m_tp_pca, m_rfp_pca, metric="euclidean")

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)

    # Avoid division by zero
    min_nonzero_distance = np.min(D[D > 0]) if np.any(D > 0) else 0.1
    D_sort[D_sort == 0] = min_nonzero_distance / 20

    W = 1.0 / D_sort
    return W, idx_sort


def pca_strategy(df_tp: pd.DataFrame, df_rp: pd.DataFrame, pcis: list[tuple], rf_param: RF_PARAM_5G, k: int = 2):
    # 1. Create the full point matrix with all beam features
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(df_tp, pcis, rf_param)

    # 3. Apply PCA to reduce dimensions
    pca = PCA(n_components=0.95)
    pca.fit(m_rp_full)
    m_rp_pca = pca.transform(m_rp_full)
    m_tp_pca = pca.transform(m_tp_full)

    W_pca, idx_sort_pca = compute_weights_pca(m_rp_pca, m_tp_pca)

    W, idx_sort = compute_weights(m_rp_full, idx_rp_full, m_tp_full, idx_tp_full)

    _, errors = wknn_one(df_tp, df_rp, idx_sort_pca, W_pca, k)

    _, errors_control = wknn_one(df_tp, df_rp, idx_sort, W, k=2)

    return errors.mean(), int(m_rp_pca.shape[0] * m_rp_pca.shape[1]), errors_control.mean(), int(
        m_rp_full.shape[0] * m_rp_full.shape[1])


In [3]:
df = df_orig
all_pcis = extract_unique_npcis(df['measurements_matrix'])

data = []

n_runs = 30
for i in range(n_runs):
    print(f"\r{i}/{n_runs}      ", end="")
    df_tp, df_rp = dataset_tp_rp_split(df, 0.3, i * 420 * 69)
    res = pca_strategy(df_tp, df_rp, all_pcis, rf_param)
    data.append(res)

res_df = pd.DataFrame(data, columns=['error', 'complexity', 'error_control', 'complexity_control'])

print('mean', res_df.mean())

res_df

29/30      mean error                 4.621515e+00
complexity            7.051839e+05
error_control         4.059055e+00
complexity_control    2.372168e+06
dtype: float64


,error,complexity,error_control,complexity_control
0,4.800646,698080,4.214546,2347294
1,4.685943,709760,4.187610,2386568
2,4.784068,706880,4.210421,2376884
3,4.690726,705680,4.145839,2372849
4,4.422494,718000,3.956742,2414275
5,4.657906,704560,4.103962,2369083
6,4.704291,706080,4.092484,2374194
7,4.539731,703120,3.973685,2364241
8,4.699841,706240,4.284458,2374732
9,4.606371,704720,3.963657,2369621


In [4]:
res_df.to_csv('pca_data_2.csv')

In [5]:
print(f"""
error {res_df['error'].mean():.2f}
complexity {res_df['complexity'].mean() / 1000:.0f} K

error control {res_df['error_control'].mean():.2f}
complexity control {res_df['complexity_control'].mean() / 1000:.0f} k
""")


error 4.62
complexity 705 K

error control 4.06
complexity control 2372 k

